In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from tqdm import tqdm
from sklearn.cluster import DBSCAN

In [2]:
milk = pd.read_csv("Datasets/milk.csv", index_col=0)

scaler = StandardScaler().set_output(transform='pandas')
df_scaled = scaler.fit_transform(milk)

In [3]:
dbscan = DBSCAN(eps=0.4, min_samples=2)
dbscan.fit(df_scaled)
df_copy = milk.copy()
df_copy['Cluster']= dbscan.labels_
df_copy

# -1 is outlier

,water,protein,fat,lactose,ash,Cluster
Animal,,,,,,
HORSE,90.1,2.6,1.0,6.9,0.35,-1
ORANGUTAN,88.5,1.4,3.5,6.0,0.24,0
MONKEY,88.4,2.2,2.7,6.4,0.18,0
DONKEY,90.3,1.7,1.4,6.2,0.40,-1
HIPPO,90.4,0.6,4.5,4.4,0.10,-1
CAMEL,87.7,3.5,3.4,4.8,0.71,1
BISON,86.9,4.8,1.7,5.7,0.90,1
BUFFALO,82.1,5.9,7.9,4.7,0.78,2
GUINEA PIG,81.9,7.4,7.2,2.7,0.85,-1


In [4]:
silhouette_score(df_scaled, dbscan.labels_)

-0.04604359194982988

In [5]:
df_scaled_inliner = df_scaled.copy()
df_scaled_inliner['Cluster']=dbscan.labels_
df_scaled_inliner = df_scaled_inliner[df_scaled_inliner["Cluster"] != -1]
df_scaled_inliner

,water,protein,fat,lactose,ash,Cluster
Animal,,,,,,
ORANGUTAN,0.821407,-1.344603,-0.660619,1.040773,-1.259945,0
MONKEY,0.813445,-1.121062,-0.738247,1.263637,-1.381249,0
CAMEL,0.757707,-0.757806,-0.670322,0.372182,-0.309730,1
BISON,0.694008,-0.394551,-0.835283,0.873626,0.074400,1
BUFFALO,0.311810,-0.087181,-0.233662,0.316466,-0.168208,2
FOX,0.271998,0.108418,-0.427733,0.427898,0.135052,2
LLAMA,0.662158,-0.646036,-0.689729,0.817910,-0.127774,1
ZEBRA,0.638270,-0.897520,-0.534472,0.650762,-0.329947,1
SHEEP,0.303847,-0.171009,-0.379215,0.316466,0.094617,2


In [6]:
len(df_scaled_inliner['Cluster'].unique())

4

In [7]:
silhouette_score(df_scaled_inliner.drop('Cluster', axis=1), df_scaled_inliner['Cluster'])

0.6518937593821538

#### Trying diff combinations of `eps` and `min_samples`

In [15]:
eps = np.linspace(0.01,2, 100)
min_samp = [2,3,4,5]
scores = []
for e in eps:
    for m in min_samp:
        dbscan = DBSCAN(eps = e, min_samples = m)
        dbscan.fit(df_scaled)
        
        df_scaled_inliner = df_scaled.copy()
        df_scaled_inliner['Cluster']=dbscan.labels_
        df_scaled_inliner = df_scaled_inliner[df_scaled_inliner["Cluster"] != -1]
        
        if  len(df_scaled_inliner['Cluster'].unique()) >= 2:
            scores.append([e, m, silhouette_score(df_scaled_inliner.drop('Cluster', axis=1), 
                                                  df_scaled_inliner['Cluster'])])

df_scores = pd.DataFrame(scores, columns=['epsilon', 'sample', 'scores'])
df_scores.sort_values('scores', ascending=False)

,epsilon,sample,scores
0,0.311515,2,0.917544
1,0.331616,2,0.903367
2,0.351717,2,0.864756
65,0.854242,3,0.657551
73,0.934646,3,0.657551
...,...,...,...
78,0.994949,2,0.434482
53,0.773838,3,0.418695
46,0.713535,3,0.418695
48,0.733636,3,0.418695


In [17]:
dbscan = DBSCAN(eps = 0.311515, min_samples=2)
dbscan.fit(df_scaled)
dbscan.labels_   # -1 are too much try diff values

array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0, -1, -1, -1, -1,  0, -1,
       -1, -1, -1,  1,  1, -1, -1, -1])

In [33]:
df_scores.iloc[15:25,:]

,epsilon,sample,scores
15,0.391919,2,-0.063546
16,0.391919,3,-0.184122
17,0.412020,2,-0.046044
18,0.412020,3,-0.105101
19,0.432121,2,0.052456
20,0.432121,3,0.102494
21,0.452222,2,0.052456
22,0.452222,3,0.102494
23,0.472323,2,0.052456
24,0.472323,3,0.102494


In [18]:
dbscan = DBSCAN(eps = 0.432121, min_samples=3)
dbscan.fit(df_scaled)
dbscan.labels_ 

array([-1,  0,  0,  0, -1,  2,  2,  1, -1, -1,  1,  2,  0, -1,  2,  1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1])

#### Using ipython widgets

In [36]:
from ipywidgets import widgets, interact

In [37]:
def dbscan_clust(e,m):
    dbscan = DBSCAN(eps = e, min_samples=m)
    dbscan.fit(df_scaled)
    return dbscan.labels_ 
    

In [38]:
interact(dbscan_clust, e = widgets.FloatSlider(min = 0.01, max = 2, step = 0.01, value = 0.3),
                        m = widgets.IntSlider(min = 2, max = 5, step =1, value = 2))

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, min=0.01, step=0.01), IntSlider(value=2…

<function __main__.dbscan_clust(e, m)>

In [40]:
interact(dbscan_clust, e = widgets.FloatSlider(min = 0.01, max = 2, step = 0.01, value = 0.3),
                        m = [2,3,4,5])

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, min=0.01, step=0.01), Dropdown(descript…

<function __main__.dbscan_clust(e, m)>

In [39]:
# Using Manhattan

eps = np.linspace(0.01,2, 100)
min_samp = [2,3,4,5]
scores = []
for e in eps:
    for m in min_samp:
        dbscan = DBSCAN(eps = e, min_samples = m, metric='manhattan')
        dbscan.fit(df_scaled)
        
        df_scaled_inliner = df_scaled.copy()
        df_scaled_inliner['Cluster']=dbscan.labels_
        df_scaled_inliner = df_scaled_inliner[df_scaled_inliner["Cluster"] != -1]
        
        if  len(df_scaled_inliner['Cluster'].unique()) >= 2:
            scores.append([e, m, silhouette_score(df_scaled_inliner.drop('Cluster', axis=1), 
                                                  df_scaled_inliner['Cluster'])])

df_scores = pd.DataFrame(scores, columns=['epsilon', 'sample', 'scores'])
df_scores.sort_values('scores', ascending=False)

,epsilon,sample,scores
0,0.512525,2,0.903367
1,0.532626,2,0.903367
2,0.552727,2,0.903367
3,0.572828,2,0.903367
4,0.592929,2,0.903367
...,...,...,...
156,1.839192,2,0.393804
148,1.758788,2,0.393804
166,1.939697,2,0.393804
162,1.899495,2,0.393804
